# 06 — Final Model Selection (Phase ML-5)

**Fresh, previously-unseen 20% test split** (`random_state=2026`), distinct
from the ML-3/4 split (`random_state=42`) that had been inspected multiple
times during earlier feature selection (see `reports/ml_final_audit.md`).

**Rule enforced throughout this notebook: the new test set is loaded once
in cell 2 for provenance, and not referenced again until the single
"FINAL EVALUATION" section near the end.** Every cell before that section
uses `dev_df` only.

Feature schemas are **frozen** (approved architecture, not re-derived):
- **Basic:** Age, Sex, Height, Weight, BP, Family H/O, Hypertension,
  Diabetes, H/O ChestPain (9 fields). BMI computed for display only.
- **Enhanced:** Basic + Total Cholesterol, LDL, Triglycerides, RBS (13
  fields total).
- **Advanced:** research/sensitivity-analysis only, not a deployment tier.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, brier_score_loss, roc_curve, make_scorer)
from xgboost import XGBClassifier
import joblib

RANDOM_STATE = 2026
TARGET = 'Heart Disease'
pd.set_option('display.width', 160)

## 1. Rebuild the cleaned adult population (same cleaning as notebooks 01-02) and create the NEW split

In [2]:
DATA_PATH = "../Resource files/Heart_diasease_dataset_from_Northern_Bangladesh.xlsx"
df = pd.read_excel(DATA_PATH, sheet_name="Our Dataset")
df.columns = [c.strip() for c in df.columns]

def clean_numeric_text(series):
    def parse(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip().replace('`', '').replace(' ', '').replace(',', '.')
        try:
            return float(s)
        except ValueError:
            return np.nan
    return series.apply(parse)

for col in ['Himoglobin', 'Potassium', 'Chloride']:
    df[col] = clean_numeric_text(df[col])

def parse_troponin(v):
    if pd.isna(v):
        return np.nan, 0
    s = str(v).strip()
    if s.startswith('>') or s.startswith('<'):
        return float(s[1:]), 1
    try:
        return float(s), 0
    except ValueError:
        return np.nan, 0

parsed = df['Troponin-I'].apply(parse_troponin)
df['troponin_raw'] = parsed.apply(lambda t: t[0])
df['Troponin_Censored'] = parsed.apply(lambda t: t[1])
is_hs = df['Troponin- I assay type'] == 'High-Sensitivity Troponin-I (ng/L)'
df['Troponin_I_harmonised'] = df['troponin_raw']
df.loc[is_hs, 'Troponin_I_harmonised'] = df.loc[is_hs, 'troponin_raw'] / 1000.0

df_adult = df.loc[df['Age'] >= 18].reset_index(drop=True)
assert len(df_adult) == 1035
print("Adult population:", df_adult.shape)

dev_df, test_df = train_test_split(df_adult, test_size=0.20, stratify=df_adult[TARGET], random_state=RANDOM_STATE)
print(f"NEW split (seed={RANDOM_STATE}): dev={dev_df.shape}, test={test_df.shape}")

_, old_test_df = train_test_split(df_adult, test_size=0.20, stratify=df_adult[TARGET], random_state=42)
overlap = len(set(test_df.index) & set(old_test_df.index))
print(f"Row overlap with the OLD (seed=42) test set: {overlap}/{len(test_df)} - confirms this is a genuinely different split.")

dev_df.to_pickle("ml5_dev.pkl")
test_df.to_pickle("ml5_test.pkl")

Adult population: (1035, 30)
NEW split (seed=2026): dev=(828, 30), test=(207, 30)
Row overlap with the OLD (seed=42) test set: 44/207 - confirms this is a genuinely different split.


## 2. Frozen feature-set definitions

In [3]:
BASIC_NUMERIC = ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)']
BASIC_BINARY = ['Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain']
BASIC_CATEGORICAL = ['Sex']
ENHANCED_NUMERIC = BASIC_NUMERIC + ['Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)']
ADVANCED_NUMERIC = ENHANCED_NUMERIC + ['Troponin_I_harmonised', 'Sodium(mmol/L)', 'Potassium', 'Chloride', 'Creatinine(mg/dL)', 'Platelets', 'Himoglobin']
ADVANCED_BINARY = BASIC_BINARY + ['Troponin_Censored']

FEATURE_SETS = {
    'A_Basic': {'numeric': BASIC_NUMERIC, 'binary': BASIC_BINARY, 'categorical': BASIC_CATEGORICAL},
    'B_Enhanced': {'numeric': ENHANCED_NUMERIC, 'binary': BASIC_BINARY, 'categorical': BASIC_CATEGORICAL},
    'C_Advanced_research': {'numeric': ADVANCED_NUMERIC, 'binary': ADVANCED_BINARY, 'categorical': BASIC_CATEGORICAL},
}
print("Excluded from all feature sets (never used as ML inputs): SL, UNIT, Troponin- I assay type")
for k, v in FEATURE_SETS.items():
    print(k, '->', v['numeric'] + v['binary'] + v['categorical'])

Excluded from all feature sets (never used as ML inputs): SL, UNIT, Troponin- I assay type
A_Basic -> ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)', 'Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain', 'Sex']
B_Enhanced -> ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)', 'Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain', 'Sex']
C_Advanced_research -> ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)', 'Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'Troponin_I_harmonised', 'Sodium(mmol/L)', 'Potassium', 'Chloride', 'Creatinine(mg/dL)', 'Platelets', 'Himoglobin', 'Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain', 'Troponin_Censored', 'Sex']


## 3. Leakage-safe preprocessing pipeline (fit within dev-set CV folds only)

In [4]:
def make_pipeline(fs_or_cols, model, binary=None, categorical=None):
    if isinstance(fs_or_cols, dict):
        numeric_cols, binary_cols, cat_cols = fs_or_cols['numeric'], fs_or_cols['binary'], fs_or_cols['categorical']
    else:
        numeric_cols, binary_cols, cat_cols = fs_or_cols, binary or BASIC_BINARY, categorical or BASIC_CATEGORICAL
    numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())])
    binary_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent'))])
    cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
    pre = ColumnTransformer([('num', numeric_pipe, numeric_cols), ('bin', binary_pipe, binary_cols), ('cat', cat_pipe, cat_cols)])
    return Pipeline([('preprocess', pre), ('model', model)])

def make_frozen_pipeline(numeric_cols, algo):
    spw = (dev_df[TARGET]==0).sum() / (dev_df[TARGET]==1).sum()
    if algo == 'xgb':
        model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=spw,
                               n_estimators=200, max_depth=3, learning_rate=0.05, n_jobs=-1)
    else:
        model = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced',
                                        n_estimators=400, max_depth=10, max_features='sqrt', n_jobs=-1)
    return make_pipeline(numeric_cols, model, BASIC_BINARY, BASIC_CATEGORICAL)

def specificity(y_true, y_pred):
    tn = ((y_true==0)&(y_pred==0)).sum(); fp = ((y_true==0)&(y_pred==1)).sum()
    return tn/(tn+fp)

## 4. Algorithm/feature-set comparison — 6 algorithms x 3 feature sets, DEV-SET 5-fold CV only

CLAUDE.md requires Decision Tree; this brief asks for LR/RF/XGBoost/SVM/KNN
— all 6 are run, as in ML-3/4, for the documented comparison.

In [5]:
neg, pos = (dev_df[TARGET]==0).sum(), (dev_df[TARGET]==1).sum()
scale_pos_weight = neg / pos

ALGORITHMS = {
    'LogisticRegression': (LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE), {'model__C': [0.01, 0.1, 1, 10]}),
    'DecisionTree': (DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE), {'model__max_depth': [3, 5, 8, None], 'model__min_samples_leaf': [1, 5, 10]}),
    'RandomForest': (RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), {'model__n_estimators': [200, 400], 'model__max_depth': [6, 10, None], 'model__max_features': ['sqrt', 'log2']}),
    'XGBoost': (XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=scale_pos_weight, n_jobs=-1), {'model__n_estimators': [200, 400], 'model__max_depth': [3, 5], 'model__learning_rate': [0.05, 0.1]}),
    'SVM': (SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE), {'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 'auto']}),
    'KNN': (KNeighborsClassifier(), {'model__n_neighbors': [5, 11, 15, 21], 'model__weights': ['uniform', 'distance']}),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_dev, y_dev = dev_df, dev_df[TARGET]

all_results = []
best_estimators = {}
t0 = time.time()
for fs_name, fs in FEATURE_SETS.items():
    for algo_name, (model, grid) in ALGORITHMS.items():
        pipe = make_pipeline(fs, model)
        gs = GridSearchCV(pipe, grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
        gs.fit(X_dev, y_dev)
        idx = gs.best_index_
        all_results.append({'feature_set': fs_name, 'algorithm': algo_name, 'best_params': gs.best_params_,
                             'cv_mean_auc': gs.cv_results_['mean_test_score'][idx], 'cv_sd_auc': gs.cv_results_['std_test_score'][idx]})
        best_estimators[(fs_name, algo_name)] = gs.best_estimator_
print(f"Total wall time: {time.time()-t0:.1f}s")
results_df = pd.DataFrame(all_results)
pivot = results_df.pivot(index='algorithm', columns='feature_set', values='cv_mean_auc')[['A_Basic','B_Enhanced','C_Advanced_research']]
pivot.round(4)

Total wall time: 193.7s


feature_set,A_Basic,B_Enhanced,C_Advanced_research
algorithm,,,
DecisionTree,0.8595,0.9303,0.9528
KNN,0.9110,0.9724,0.9789
LogisticRegression,0.9014,0.9790,0.9912
RandomForest,0.9327,0.9853,0.9978
SVM,0.9299,0.9836,0.9906
XGBoost,0.9386,0.9830,0.9964


## 5. Paired significance test: Random Forest vs XGBoost (frozen Basic/Enhanced, dev-only)

In [6]:
for fs_name, cols in [('A_Basic', BASIC_NUMERIC), ('B_Enhanced', ENHANCED_NUMERIC)]:
    xgb_scores = cross_val_score(make_frozen_pipeline(cols, 'xgb'), dev_df, dev_df[TARGET], cv=cv, scoring='roc_auc')
    rf_scores = cross_val_score(make_frozen_pipeline(cols, 'rf'), dev_df, dev_df[TARGET], cv=cv, scoring='roc_auc')
    t_stat, p_value = stats.ttest_rel(xgb_scores, rf_scores)
    print(f"{fs_name}: XGB={np.round(xgb_scores,4)}  RF={np.round(rf_scores,4)}  paired t-test t={t_stat:.3f} p={p_value:.4f}")
print("\nNo statistically significant difference found (see p-values above) -> 'comparable performance', not 'statistically equivalent'.")

A_Basic: XGB=[0.9519 0.9177 0.9446 0.9141 0.9476]  RF=[0.9441 0.9094 0.9481 0.9129 0.9486]  paired t-test t=1.069 p=0.3454


B_Enhanced: XGB=[0.979  0.9781 0.99   0.9748 0.9933]  RF=[0.9801 0.9811 0.9879 0.9827 0.9949]  paired t-test t=-1.409 p=0.2316

No statistically significant difference found (see p-values above) -> 'comparable performance', not 'statistically equivalent'.


## 6. Threshold selection — out-of-fold dev predictions only (test set not touched)

In [7]:
FROZEN_THRESHOLDS = {}
for fs_name, cols in [('A_Basic', BASIC_NUMERIC), ('B_Enhanced', ENHANCED_NUMERIC)]:
    pipe = make_frozen_pipeline(cols, 'xgb')
    oof_proba = cross_val_predict(pipe, dev_df, dev_df[TARGET], cv=cv, method='predict_proba')[:, 1]
    y_true = dev_df[TARGET].values
    fpr, tpr, thresh = roc_curve(y_true, oof_proba)
    spec = 1 - fpr
    youden_idx = np.argmax(tpr - fpr)
    print(f"\n{fs_name}: OOF AUC={roc_auc_score(y_true, oof_proba):.4f}  OOF Brier={brier_score_loss(y_true, oof_proba):.4f}")
    for label, idx in [('default 0.5', np.argmin(np.abs(thresh-0.5))), ('Youden-optimal', youden_idx)]:
        print(f"  {label:16s} thresh={thresh[idx]:.4f} sens={tpr[idx]:.4f} spec={spec[idx]:.4f}")
    for target in [0.90, 0.95]:
        candidates = [(t,tp,sp) for t,tp,sp in zip(thresh,tpr,spec) if tp>=target]
        if candidates:
            t_,tp_,sp_ = max(candidates, key=lambda x: x[0])
            print(f"  sens-target>={target:.2f}   thresh={t_:.4f} sens={tp_:.4f} spec={sp_:.4f}")
    np.save(f"ml5_oof_proba_{fs_name}.npy", oof_proba)
    np.save(f"ml5_oof_y_{fs_name}.npy", y_true)


A_Basic: OOF AUC=0.9357  OOF Brier=0.1015
  default 0.5      thresh=0.4999 sens=0.8355 spec=0.8917
  Youden-optimal   thresh=0.4681 sens=0.8483 spec=0.8833
  sens-target>=0.90   thresh=0.3224 sens=0.9017 spec=0.7833
  sens-target>=0.95   thresh=0.1796 sens=0.9509 spec=0.6028



B_Enhanced: OOF AUC=0.9822  OOF Brier=0.0533
  default 0.5      thresh=0.4991 sens=0.9124 spec=0.9417
  Youden-optimal   thresh=0.3844 sens=0.9444 spec=0.9167
  sens-target>=0.90   thresh=0.5583 sens=0.9060 spec=0.9472
  sens-target>=0.95   thresh=0.3335 sens=0.9509 spec=0.9028


**Frozen threshold decisions (reasoned per tier, not mechanically
identical):**
- **A_Basic -> Youden-optimal (0.4681).** Forcing 95% sensitivity on the
  weaker Basic-only model costs specificity down to ~0.60 (too many false
  positives for a first-pass low-burden screen) - the balanced Youden point
  (sens 0.85 / spec 0.88) is used instead.
- **B_Enhanced -> sensitivity-target 95% (0.3335).** The stronger Enhanced
  model affords a high-sensitivity operating point while keeping
  specificity reasonable (~0.90) - consistent with prioritising sensitivity
  for a screening-support tool once there's enough signal to support it.

## 7. Calibration analysis (Brier score + reliability diagram), dev-OOF (primary) and final test (corroborating)

Brier score alone does not prove calibration (combines discrimination +
calibration) - the reliability diagrams below are the actual calibration
check.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, fs_name in zip(axes, ['A_Basic', 'B_Enhanced']):
    proba = np.load(f"ml5_oof_proba_{fs_name}.npy")
    y = np.load(f"ml5_oof_y_{fs_name}.npy")
    frac_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy='quantile')
    ax.plot([0,1],[0,1],'k--', label='Perfect calibration')
    ax.plot(mean_pred, frac_pos, 'o-', label='XGBoost (dev OOF)')
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed frequency')
    ax.set_title(f'{fs_name} — dev-set (OOF) calibration, n=828'); ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/ml5_calibration_dev_oof.png", dpi=120)
plt.show()
print("Dev-set (n=828) reliability diagrams are the primary calibration evidence - larger, less noisy sample than the n=207 test set.")

Dev-set (n=828) reliability diagrams are the primary calibration evidence - larger, less noisy sample than the n=207 test set.


## 8. FINAL EVALUATION — the new test set is used starting here, exactly once, after every decision above was frozen

In [9]:
print("Loading test_df for the first and only decision-relevant evaluation in this notebook.")
test_df = pd.read_pickle("ml5_test.pkl")
print("Test shape:", test_df.shape)

FROZEN = {
    'A_Basic': {'numeric': BASIC_NUMERIC, 'threshold': 0.4681},
    'B_Enhanced': {'numeric': ENHANCED_NUMERIC, 'threshold': 0.3335},
}
Path("../artifacts").mkdir(exist_ok=True)
final_results = {}
fig2, axes2 = plt.subplots(1, 2, figsize=(11, 5))
for ax, (fs_name, cfg) in zip(axes2, FROZEN.items()):
    print(f"\n{'='*70}\n{fs_name}\n{'='*70}")
    for algo, label, thresh in [('xgb', 'XGBoost (PRIMARY)', cfg['threshold']), ('rf', 'RandomForest (alternative, @0.5)', 0.5)]:
        pipe = make_frozen_pipeline(cfg['numeric'], algo)
        pipe.fit(dev_df, dev_df[TARGET])
        proba = pipe.predict_proba(test_df)[:, 1]
        pred = (proba >= thresh).astype(int)
        y = test_df[TARGET].values
        metrics = {'roc_auc': roc_auc_score(y,proba), 'accuracy': accuracy_score(y,pred),
                   'precision': precision_score(y,pred), 'recall': recall_score(y,pred),
                   'specificity': specificity(y,pred), 'f1': f1_score(y,pred), 'brier': brier_score_loss(y,proba)}
        cm = confusion_matrix(y, pred)
        print(f"\n{label} (threshold={thresh:.4f}):")
        for k,v in metrics.items(): print(f"  {k:12s} {v:.4f}")
        print("  Confusion matrix [[TN FP][FN TP]]:\n ", cm)
        if algo == 'xgb':
            frac_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy='quantile')
            ax.plot([0,1],[0,1],'k--'); ax.plot(mean_pred, frac_pos, 's-', color='darkorange', label='XGBoost (final test)')
            ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed frequency')
            ax.set_title(f'{fs_name} — final test calibration, n=207'); ax.legend()
            joblib.dump(pipe, f"../artifacts/{fs_name.lower()}_xgboost_pipeline.joblib")
        final_results.setdefault(fs_name, {})[algo] = {'metrics': metrics, 'cm': cm.tolist()}
plt.tight_layout()
plt.savefig("../reports/figures/ml5_calibration_test.png", dpi=120)
plt.show()

majority = dev_df[TARGET].mode()[0]
print(f"\nMajority-class baseline test accuracy: {accuracy_score(test_df[TARGET], np.full(len(test_df), majority)):.4f}")
print("\nFinal XGBoost pipelines saved to ../artifacts/.")

Loading test_df for the first and only decision-relevant evaluation in this notebook.
Test shape: (207, 30)

A_Basic



XGBoost (PRIMARY) (threshold=0.4681):
  roc_auc      0.9135
  accuracy     0.8454
  precision    0.8829
  recall       0.8376
  specificity  0.8556
  f1           0.8596
  brier        0.1121
  Confusion matrix [[TN FP][FN TP]]:
  [[77 13]
 [19 98]]



RandomForest (alternative, @0.5) (threshold=0.5000):
  roc_auc      0.9102
  accuracy     0.8502
  precision    0.8909
  recall       0.8376
  specificity  0.8667
  f1           0.8634
  brier        0.1178
  Confusion matrix [[TN FP][FN TP]]:
  [[78 12]
 [19 98]]

B_Enhanced



XGBoost (PRIMARY) (threshold=0.3335):
  roc_auc      0.9787
  accuracy     0.9420
  precision    0.9412
  recall       0.9573
  specificity  0.9222
  f1           0.9492
  brier        0.0487
  Confusion matrix [[TN FP][FN TP]]:
  [[ 83   7]
 [  5 112]]



RandomForest (alternative, @0.5) (threshold=0.5000):
  roc_auc      0.9777
  accuracy     0.9469
  precision    0.9732
  recall       0.9316
  specificity  0.9667
  f1           0.9520
  brier        0.0540
  Confusion matrix [[TN FP][FN TP]]:
  [[ 87   3]
 [  8 109]]



Majority-class baseline test accuracy: 0.5652

Final XGBoost pipelines saved to ../artifacts/.


## 9. Summary

See `../reports/ml5_final_model_selection.md` for the full written report
(final architecture, all 17 required sections, limitations, and the
explicit statement that this test set was not used for any model or
feature selection decision above).